In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import dask
import dask.distributed
import seaborn as sns
import rasterio
import os

import xarray as xr
import rioxarray
import cf_xarray
import xcdat
from rioxarray.merge import merge_arrays
from matplotlib import cm

## Use this to georeference mask PNGs with their GeoTIFF counterpart (must be same dims)

In [2]:
def georeference_mask_pngs(image_dir, mask_dir, output_dir, image_dimensions=1024):
    images = sorted(os.listdir(image_dir))
    masks = sorted(os.listdir(mask_dir))
    for item in range(0,len(masks)):
        wt_mask = rioxarray.open_rasterio(fr'{mask_dir}/{masks[item]}')
        wt_image = rioxarray.open_rasterio(fr'{image_dir}/{images[item]}')
        wt_mask_ds = wt_mask.to_dataset(dim='band')
        wt_image_ds = wt_image.to_dataset(dim='band')

        minx, miny, maxx, maxy = tuple(wt_image.rio.bounds())
        x_coords = np.linspace(minx, maxx, image_dimensions)
        y_coords = np.linspace(maxy, miny, image_dimensions)
        wt_mask_ds = wt_mask_ds.assign_coords(x=x_coords, y=y_coords).rio.write_crs('EPSG:3413')
        wt_mask_ds = wt_mask_ds.rio.reproject_match(wt_image)

        wt_image_ds['mask'] = (('y','x'), wt_mask_ds[1].data)
        wt_image_ds['mask'].rio.to_raster(fr'{output_dir}/{masks[item][:-4]}_georef.tif', driver='GTiff')

#Using my own image conventions where source images dataset is not one-to-one with masks; need to make script to filter tifs before the pngs...
def georeference_mask_pngs_specialized(image_dir, mask_dir, output_dir, image_dimensions=1024):
    images = sorted(os.listdir(image_dir))
    masks = sorted(os.listdir(mask_dir))
    image_dictionary = {}
    for item in images:
        image_dictionary[(item[-9:-4] + '_0_predseg.png')] = item
    for item in range(0,len(masks)):
        wt_mask = rioxarray.open_rasterio(fr'{mask_dir}/{masks[item]}')
        wt_image = rioxarray.open_rasterio(fr'{image_dir}/{image_dictionary[masks[item]]}')
        wt_mask_ds = wt_mask.to_dataset(dim='band')
        wt_image_ds = wt_image.to_dataset(dim='band')

        minx, miny, maxx, maxy = tuple(wt_image.rio.bounds())
        x_coords = np.linspace(minx, maxx, image_dimensions)
        y_coords = np.linspace(maxy, miny, image_dimensions)
        wt_mask_ds = wt_mask_ds.assign_coords(x=x_coords, y=y_coords).rio.write_crs('EPSG:3413')
        wt_mask_ds = wt_mask_ds.rio.reproject_match(wt_image)

        wt_image_ds['mask'] = (('y','x'), wt_mask_ds[1].data)
        wt_image_ds['mask'].rio.to_raster(fr'{output_dir}/{masks[item][:-4]}_georef.tif', driver='GTiff')



In [4]:
image_dir = fr'D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_Total'
mask_dir = fr'D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_ML_Mask_Test\1024_from_Model'
output_dir = fr'D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Images_WV02_20180726231451\Sections_ML_Mask_Test\1024_from_Model_georef'

georeference_mask_pngs_specialized(image_dir=image_dir, mask_dir=mask_dir, output_dir=output_dir)

c:\Users\gaimholte\AppData\Local\miniforge3\envs\geol437_plus\Lib\site-packages\rioxarray\_io.py:1148: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  warnings.warn(str(rio_warning.message), type(rio_warning.message))  # type: ignore
c:\Users\gaimholte\AppData\Local\miniforge3\envs\geol437_plus\Lib\site-packages\rioxarray\_io.py:1148: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  warnings.warn(str(rio_warning.message), type(rio_warning.message))  # type: ignore
c:\Users\gaimholte\AppData\Local\miniforge3\envs\geol437_plus\Lib\site-packages\rioxarray\_io.py:1148: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  warnings.warn(str(rio_warning.message), type(rio_warning.message))  # type: ignore
c:\Users\gaimholte\AppData\Local\miniforge3\envs\geol437_plus\Lib\site-packages\rioxarray\_io.py:1148: NotGeorefere

In [4]:
thing = os.listdir(fr"D:\Imholte_Research_WT\High_Res_WT_Images_and_Masks\Images\Full_Images_Masks_and_Sections_WV02_20190820222750\Sections_Total")[0]
thing

'WV02_20190820222750_10300100968FB300_19AUG20222750-M1BS-503550221040_01_P004_u16rf3413_01_01.tif'

In [13]:
thing[-9:-4] + '_0_predseg.png'

'01_01_0_predseg.png'